
#### Stage 1

In this stage, we are trying to predict whether the customer will buy in their current session or not. 

In [ ]:
#import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
#features selection and class imbalance
from sklearn.feature_selection import RFE, VarianceThreshold
from sklearn.kernel_approximation import RBFSampler
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import PolynomialCountSketch
from sklearn.pipeline import Pipeline, make_pipeline
from imblearn.over_sampling import SMOTE
#machine learning models
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier
#model performance evaluation
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.metrics import ConfusionMatrixDisplay, roc_auc_score, roc_curve, confusion_matrix
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
train = pd.read_csv('final_session_level_train.csv')
test = pd.read_csv('final_session_level_test.csv')

In [ ]:
train.head(10)

When inspecting the dataset, we observed that we need to transform the buy_flag feature to numerical values before running our model. We also dropped columns that will not be needed later on. 

In [ ]:
#preprocessing data
buy = pd.get_dummies(train['buy_flag'], drop_first=True) 
train = pd.concat([train, buy], axis=1)
train.columns = [*train.columns[:-1], 'Buy']

buy = pd.get_dummies(test['buy_flag'], drop_first=True) 
test = pd.concat([test, buy], axis=1)
test.columns = [*test.columns[:-1], 'Buy']

rmv_col = ['buy_flag', 'Unnamed: 0', 'Session ID']
train = train.drop(rmv_col, axis=1)
test = test.drop(rmv_col, axis=1)

## Full Dataset

As observed below, our data is highly imbalanced, with 95% of data classified as class 0 and 5% of data classified as class 1. We will need to address this when running our model, either by specifying in the parameters or performing different sampling techniques, such as over-sampling or under-sampling to balance the data.

In [ ]:
sns.countplot(x = 'Buy', data=train)

Here, we will split the full dataset into training and testing set. 

In [ ]:
X_train = train.drop("Buy", axis=1)
y_train = train["Buy"]

X_test = test.drop("Buy", axis=1)
y_test = test["Buy"] 

We will compare the performances of 8 different models: Logistic Regression, XGBoost, Support Vector Machine (SVM), Neural Networks, CatBoost, Light GBM, and Stack Generalizer on the sample dataset. On the full dataset, we will implement Logistic Regression, XGBoost, CatBoost, and SVM. 

### 1. Logistic Regression

Here, we will train the Logistic Regression model on the entire dataset.

In [ ]:
logreg = LogisticRegression()
rfe = RFE(logreg, 20)
rfe = rfe.fit(X_train, y_train.values.ravel())
print(rfe.support_)
print(rfe.ranking_)

In [ ]:
logit_model = sm.Logit(y_train,X_train)
result = logit_model.fit()
print(result.summary2())

As unique_items has a p-value greater than 0.05, which indicates that this feature is not significant, we will drop it from our dataset.

In [ ]:
X_train = X_train.drop("unique_items", axis=1)
X_test= X_test.drop("unique_items", axis=1)

We apply SMOTE to handle class imbalance before training our model. 

In [ ]:
os = SMOTE(random_state = 42, n_jobs=-1)
columns = X_train.columns

os_data_X, os_data_y = os.fit_resample(X_train, y_train)
os_data_X = pd.DataFrame(data = os_data_X,columns = columns)
os_data_y = pd.DataFrame(data = os_data_y,columns = ['Buy'])

print("length of oversampled data is ",len(os_data_X))
print("Number of no default in oversampled data",len(os_data_y[os_data_y['Buy'] == 0]))
print("Number of default",len(os_data_y[os_data_y['Buy'] == 1]))
print("Proportion of no default data in oversampled data is ",len(os_data_y[os_data_y['Buy'] == 0])/len(os_data_X))
print("Proportion of default data in oversampled data is ",len(os_data_y[os_data_y['Buy'] == 1])/len(os_data_X))

In [ ]:
logreg = LogisticRegression()
logreg.fit(os_data_X, os_data_y)

In [ ]:
y_pred = logreg.predict(X_test)

We generate the ROC Curve to find the optimal threshold where our model will achieve highest performance.

In [ ]:
logit_roc_auc = roc_auc_score(y_test, logreg.predict(X_test))
fpr, tpr, thresholds = roc_curve(y_test, logreg.predict_proba(X_test)[:,1])
plt.figure()
plt.plot(fpr, tpr, label='Logistic Regression (area = %0.2f)' % logit_roc_auc)
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()

In [ ]:
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print("Threshold value is:", optimal_threshold)

In [ ]:
THRESHOLD = optimal_threshold
preds = np.where(logreg.predict_proba(X_test)[:,1] > THRESHOLD, 1, 0)

tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
(tn, fp, fn, tp)

In [ ]:
print(classification_report(y_test, preds))

### 2. XGBoost

Here, we applied XGBoost to the full dataset. Ideally, we would want to tune the following parameters: 
1. *n_estimators*: number of estimators when boosting is terminated. 
2. *max_depth*: maximum depth of a tree. Higher value can make the tree overfit
3. *gamma*: minimum loss reduction required to make partition on a leaf node of the tree. the higher value of gamma, the more conservative the algorithm will be

However, since *GridSearchCV* performs exhaustive search over specified parameter values, it will take hours, if not days, for it to finish tuning our parameters given our laptop are not of the professionals. Therefore, in the interest of time, we will specify these parameters manually. Below is the codes for which we will use to tune our parameters.

In [ ]:
## XGB - number of estimators
param_test = {'n_estimators': range(50, 500, 50)}
gsearch1 = GridSearchCV(estimator = XGBClassifier(objective='binary:logistic', scale_pos_weight = 18, learning_rate=0.2, 
                                                  use_label_encoder =False, eval_metric = 'auc'), 
                        param_grid = param_test, scoring='roc_auc', n_jobs=-1, cv=5)
gsearch1.fit(X_train, y_train)
gsearch1.best_params_

In [ ]:
## XGB - maximum tree depth
param_test2 = {'max_depth':range(1,10,2)}
gsearch2 = GridSearchCV(estimator = XGBClassifier(objective='binary:logistic', scale_pos_weight = 18,
                                                  n_estimators = gsearch1.best_params_['n_estimators'], 
                                                  learning_rate = 0.2, use_label_encoder =False, eval_metric = 'auc'), 
                        param_grid = param_test2, scoring='roc_auc', n_jobs=-1, cv=5)

gsearch2.fit(X_train, y_train)
gsearch2.best_params_

In [ ]:
#tuning gamma parameter
param_test3 = {'gamma': [i/10.0 for i in range(0,5)]} 
gsearch3 = GridSearchCV(estimator = XGBClassifier(objective='binary:logistic', scale_pos_weight = 18,
                                                  n_estimators = gsearch1.best_params_['n_estimators'], 
                                                  learning_rate = 0.2, use_label_encoder =False, eval_metric = 'auc',
                                                  max_depth = gsearch2.best_params_['max_depth']), 
                        param_grid = param_test3, scoring='roc_auc', n_jobs=-1, cv=5)

gsearch3.fit(X_train, y_train)
gsearch3.best_params_

In [ ]:
## Tuned model
XGB_model = XGBClassifier(objective='binary:logistic', scale_pos_weight = 18, n_estimators = gsearch1.best_params_, 
                          max_depth = gsearch2.best_params_, gamma = gsearch3.best_params_, 
                          scale_pos_weight = 17, n_jobs=-1, learning_rate = 0.2, 
                          #we need to account for imbalance class ratio 1:17.98 using scale_pos_weight
                          use_label_encoder =False, eval_metric = 'auc')

XGB_model.fit(X_train, y_train)
XGB_model_predict = XGB_model.predict(X_test)
XGB_model

We will then specify the parameters manually: 
1. *n_estimators*: 50 as a value that would work well given the power of our system
2. *max_depth*: as the default is 6 and increasing this value will make the model more complex and overfit, we will go with 7
3. *gamma*: the default value is 0 and increasing it will make the algorithm more conservative. Thus, we want to start at a smaller value like 0.1 and can alter it later on if required
4. *scale_pos_weight*: since the ratio between class 0 and 1 is 18, we will specify this value for this parameter

In [ ]:
#full dataset split
X_train = train.drop("Buy", axis=1)
y_train = train["Buy"]

X_test = test.drop("Buy", axis=1)
y_test = test["Buy"] 

In [ ]:
## Tuned model
XGB_model = XGBClassifier(objective='binary:logistic', n_estimators = 50, 
                          scale_pos_weight = 18, n_jobs=-1, learning_rate = 1, alpha = 0.1,
                          #we need to account for imbalance class ratio 1:17.98 using scale_pos_weight
                          use_label_encoder =False, eval_metric = 'auc')

XGB_model.fit(X_train, y_train)
XGB_model_predict = XGB_model.predict(X_test)
XGB_model

In [ ]:
print('XGB Test Accuracy:', accuracy_score(y_test, XGB_model_predict), '\n')
print(classification_report(y_test, XGB_model_predict))
print(confusion_matrix(y_test, XGB_model_predict), '\n')

Observing the results above, our parameters will need further tuning to be able to classify class 1 better. We chose to add a booster called 'gblinear', which uses linear functions instead of the default 'gbtree', which uses tree based model. We chose linear functions since logistic regression and SVM, which are both linear classifiers, seem to work well on this dataset. 

In [ ]:
## Tuned model
XGB_model = XGBClassifier(objective='binary:logistic', n_estimators = 50, booster = 'gblinear',
                          scale_pos_weight = 18, n_jobs=-1, learning_rate = 1, alpha = 0.1,
                          #we need to account for imbalance class ratio 1:17.98 using scale_pos_weight
                          use_label_encoder =False, eval_metric = 'auc')

XGB_model.fit(X_train, y_train)
XGB_model_predict = XGB_model.predict(X_test)
XGB_model

In [ ]:
print('XGB Test Accuracy:', accuracy_score(y_test, XGB_model_predict), '\n')
print(classification_report(y_test, XGB_model_predict))
print(confusion_matrix(y_test, XGB_model_predict), '\n')

### 3. CatBoost

Here, we applied CatBoost to the full dataset. 

In [ ]:
#full dataset split
X_train = train.drop("Buy", axis=1)
y_train = train["Buy"]

X_test = test.drop("Buy", axis=1)
y_test = test["Buy"] 

Similar to our reasoning for XGBoost above, we will manually specify the parameters for this model:
1. *iterations*: 50 as a good value for our computer performance
2. *class_weights*: given the ratio between class 0 and 1 at 1:18, we will specify this in our model

In [ ]:
ctb = CatBoostClassifier(eval_metric='AUC', learning_rate = 0.2, class_weights={0: 1, 1: 18}, iterations=50)
ctb.fit(X_train, y_train)
y_pred = ctb.predict(X_test)

In [ ]:
print(ctb.get_all_params())
print(classification_report(y_test, y_pred), '\n')

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
(tn, fp, fn, tp)

As observed above, CatBoost does not work well in the full dataset compare to its performance in the sample dataset. Additionally, it does not allow us to specify between tree based model and linear functions. Thus, we were unable to improve on its performance. This is one of the reasons why XGBoost is so popular given its flexibility in using both linear model and tree learning algorithms.

As Light GBM is a similar method to CatBoost and XGBoost, while not having the flexibility as XGBoost in specifying the linear functions. It only allows to choosing tree-based methods, such as random forest, gradient boosting decision tree, etc, as boosting types. Thus, we decided to not run it on the full dataset. 

### 4. Support Vector Machine (SVM)

Here, we apply SVM to our full dataset. Similar to our reasoning for XGBoost and CatBoost, we will manually specify the parameters for our model:
1. *max_iter*: the default maximum number of passes over training data is 1000. However, in the interest of time and computer limitations, we will use 100 
2. *loss*: we used hinge as it is a loss function typically used for training classifiers
3. *penalty*: we used l1 as our regularisation term as it might bring sparsity in our model by shrinking some feature coefficients to 0, which helps reducing its complexity while still reducing the model's variance
4. *class_weight*: even though our actual ratio is 1:18, after testing with a few weights, we found that 1:36 gives us better performance results. This means that we need to give twice the weight from the original ratio to class 1
5. *alpha*: given that the default is 0.0001 and the higher the value, the stronger the regularisation, we chose alpha value at 0.01 so that it will penalises more when there is a misclassification

In [ ]:
scaler = StandardScaler()
SVM_Xtrain = scaler.fit_transform(X_train)
SVM_Xtest = scaler.transform(X_test)
clf = SGDClassifier(loss='hinge', penalty='l1', n_jobs=-1, class_weight={0: 1, 1: 36}, max_iter=100,
                   alpha=0.01)
clf.fit(SVM_Xtrain, y_train)

In [ ]:
y_pred = clf.predict(SVM_Xtest)
print(classification_report(y_test, y_pred))
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
(tn, fp, fn, tp)

### Summary

As observed above, when implementing the models to the full dataset, the performance between each varies significantly. CatBoost does terrible, while Logistic Regression, XGBoost and SVM do much better. As Logistic Regression and XGBoost have slightly better performance than SVM, we will implement these two methods in our final results and compare the scoring. 

------------------------------------------------------------------------------------------------------------------------------

## Sample Data - Model Performance Comparison

As we have a large dataset, we will compare model performance based on a small subset of data. In this case, we will use latest month data to run our models, make predictions, and compare performances. In this case, we will split the test_data, which also is latest 1-month data, into training set and testing set using 75:25 ratio. 

Here, we also observed the highly imbalanced ratio between class 0 and class 1, which helps when training the models as it mimics the issue we have observed in the full dataset.

In [ ]:
sns.countplot(x = 'Buy', data=test)

### 1. Logistic Regression

Logistic Regression is a discriminative classifier that uses a linear regression model to represent the probability through a logistic function in matrix notation: 
\begin{equation}
p(X) = \frac{exp(w^Tx)}{1+exp(w^Tx)}
\end{equation}
where $w^T$ represents the matrix for coefficients for each feature and x represents the observations. 

Logistic regression is popular machine learning method for binary classification. While simple, it gives good accuracy and has low training time. Additionally, it outputs a probability of the event being classified as class 0 or 1, which can then be adjusted using different threshold values depending on our classification purpose. For example, if the cost of misclassifying class 1 is too high, we would want to have a higher recall rate for class 1, thus we can do so by increasing the threshold value for the model to classify obsevation as class 1.  

In [ ]:
#splitting data into training and test set
X = test.drop("Buy", axis=1)
y = test["Buy"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

While there are many features selection method, we use Recursive Feature Elimination method, which selects the most significant features by recursively considering smaller sets of features. The least important features are pruned from the current set of features and this process is repeated. 

In [ ]:
logreg = LogisticRegression()
rfe = RFE(logreg, 20)
rfe = rfe.fit(X_train, y_train.values.ravel())
print(rfe.support_)
print(rfe.ranking_)

From the results above, all features are considered important so we will not need to remove any of them.

In [ ]:
logit_model = sm.Logit(y_train,X_train)
result = logit_model.fit()
print(result.summary2())

As Popularity has p-value greater than 0.05, it is considered insignificant, so we will remove it from our dataset. 

In [ ]:
X_train = X_train.drop("Popularity", axis=1)
X_test= X_test.drop("Popularity", axis=1)

We then address the issue of class imbalance in our dataset using the Synthetic Minority Over-Sampling Technique (SMOTE), which creates synthetic samples from minor class instead of simply creating copies. It works by randomly chooses one of the K-nearest-neighbours and use it to create similar but randomly tweaked, new observations. 

There are different opinions over whether we should eliminate features before or after SMOTE. The argument given is that SMOTE can induce bias and it violates the independence assumption as it uses KNN technique, which implies that there are relationships between these features. In our case, we know that these features are somewhat related to each other as they are generated from the same dataset, we chose to apply RFE before SMOTE.  

In [ ]:
os = SMOTE(random_state = 42, n_jobs=-1)
columns = X_train.columns

os_data_X, os_data_y = os.fit_resample(X_train, y_train)
os_data_X = pd.DataFrame(data = os_data_X,columns = columns)
os_data_y = pd.DataFrame(data = os_data_y,columns = ['Buy'])

print("length of oversampled data is ",len(os_data_X))
print("Number of no default in oversampled data",len(os_data_y[os_data_y['Buy'] == 0]))
print("Number of default",len(os_data_y[os_data_y['Buy'] == 1]))
print("Proportion of no default data in oversampled data is ",len(os_data_y[os_data_y['Buy'] == 0])/len(os_data_X))
print("Proportion of default data in oversampled data is ",len(os_data_y[os_data_y['Buy'] == 1])/len(os_data_X))

In [ ]:
logreg = LogisticRegression()
logreg.fit(os_data_X, os_data_y)

In [ ]:
y_pred = logreg.predict(X_test)

To analyse the Logistic Regression model performance, we will use the Receiver Operating Characteristic (ROC) Curve to find the optimal threshold as we look to have high recall rates for both class 0 and 1.  

In [ ]:
#Create ROC curve
logit_roc_auc = roc_auc_score(y_test, logreg.predict(X_test))
fpr, tpr, thresholds = roc_curve(y_test, logreg.predict_proba(X_test)[:,1])
plt.figure()
plt.plot(fpr, tpr, label='Logistic Regression (area = %0.2f)' % logit_roc_auc)
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()

In [ ]:
#Find optimal threshold
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print("Threshold value is:", optimal_threshold)

In [ ]:
#results based on optimal threshold
THRESHOLD = optimal_threshold
preds = np.where(logreg.predict_proba(X_test)[:,1] > THRESHOLD, 1, 0)

tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
(tn, fp, fn, tp)

In [ ]:
print(classification_report(y_test, preds))

Here, we also tried to optimise the Logistic Regression model by C or regularisation parameter, and penalty term, to see if we can further improve on the results. The larger the value of C, the higher the tolerance for misclassification. 

In [ ]:
params = {
    "C": [0.1, 1, 10, 100, 500, 1000],
    "penalty": ["l2", "l1", "elasticnet", "none"]
}
model = LogisticRegression(solver ='saga') #saga supports all penalty and faster for large dataset
logreg = GridSearchCV(model, param_grid=params, n_jobs=-1, cv=5, scoring='roc_auc', refit=True)
logreg.fit(os_data_X, os_data_y)
logreg.best_estimator_

In [ ]:
y_pred = logreg.predict(X_test)

In [ ]:
#Create ROC curve
logit_roc_auc = roc_auc_score(y_test, logreg.predict(X_test))
fpr, tpr, thresholds = roc_curve(y_test, logreg.predict_proba(X_test)[:,1])
plt.figure()
plt.plot(fpr, tpr, label='Logistic Regression (area = %0.2f)' % logit_roc_auc)
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()

In [ ]:
#find optimal threshold
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print("Threshold value is:", optimal_threshold)

In [ ]:
#results based on optimal threshold
THRESHOLD = optimal_threshold
preds = np.where(logreg.predict_proba(X_test)[:,1] > THRESHOLD, 1, 0)

tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
(tn, fp, fn, tp)

In [ ]:
print(classification_report(y_test, preds))

As observed above, the results from our parameters tuning are similar to if we just use the default parameters. Hence, this might not be needed when running on the full dataset given the time it took to tune these parameters on large dataset.

### 2. XGBoost

Next, we tried a gradiant boosting machine learning method called XGBoost (extreme Gradient Boosting), which is an implementation of gradiant boosted decision trees. Boosting is an ensemble method, where models are added consecutively to improve accuracy of the previous ones, and gradient boosting uses gradient descent algorithm to minimise the loss when adding new models. 

XGBoost is popular as it provides results with high accuracy in both classification and regression problems. Before implementing XGBoost, we will tune 4 of its parameters:
1. *n_estimators*: number of estimators when boosting is terminated. 
2. *max_depth*: maximum depth of a tree. Higher value can make the tree overfit
3. *gamma*: minimum loss reduction required to make partition on a leaf node of the tree. the higher value of gamma, the more conservative the algorithm will be

To tune these parameters, we will use *GridSearchCV()* which performs exhaustive search over specified parameter values for a model. It works by cross-validating grid-search over parameter grid.

In [ ]:
#splitting data into training and test set
X = test.drop("Buy", axis=1)
y = test["Buy"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
## XGB - number of estimators
param_test = {'n_estimators': range(1,100,4)}
gsearch1 = GridSearchCV(estimator = XGBClassifier(objective='binary:logistic', learning_rate=0.2, scale_pos_weight = 18,
                                                  use_label_encoder =False, eval_metric = 'auc'), 
                        param_grid = param_test, scoring='roc_auc', n_jobs=-1, cv=5)
gsearch1.fit(X_train, y_train)
gsearch1.best_params_

In [ ]:
## XGB - maximum tree depth
param_test2 = {'max_depth':range(1,10,2)}
gsearch2 = GridSearchCV(estimator = XGBClassifier(objective='binary:logistic', scale_pos_weight = 18,
                                                  n_estimators = gsearch1.best_params_['n_estimators'], 
                                                  learning_rate = 0.2, use_label_encoder =False, eval_metric = 'auc'), 
                        param_grid = param_test2, scoring='roc_auc', n_jobs=-1, cv=5)

gsearch2.fit(X_train, y_train)
gsearch2.best_params_

In [ ]:
#tuning gamma parameter
param_test3 = {'gamma': [i/10.0 for i in range(0,5)]} 
gsearch3 = GridSearchCV(estimator = XGBClassifier(objective='binary:logistic', scale_pos_weight = 18,
                                                  n_estimators = gsearch1.best_params_['n_estimators'], 
                                                  learning_rate = 0.2, use_label_encoder =False, eval_metric = 'auc',
                                                  max_depth = gsearch2.best_params_['max_depth']), 
                        param_grid = param_test3, scoring='roc_auc', n_jobs=-1, cv=5)

gsearch3.fit(X_train, y_train)
gsearch3.best_params_

In our final model, we will also specify the scale_post_weight value, which is the ratio between class 0 and 1 at 18.

In [ ]:
## Tuned model
XGB_model = XGBClassifier(objective='binary:logistic', n_estimators = gsearch1.best_params_['n_estimators'], 
                          max_depth = gsearch2.best_params_['max_depth'],gamma = gsearch3.best_params_['gamma'], 
                          scale_pos_weight = 18, n_jobs=-1, learning_rate = 0.2, 
                          #we need to account for imbalance class ratio 1:17.98 using scale_pos_weight
                          use_label_encoder =False, eval_metric = 'auc')

XGB_model.fit(X_train, y_train)
XGB_model_predict = XGB_model.predict(X_test)
XGB_model

In [ ]:
print('XGB Test Accuracy:', accuracy_score(y_test, XGB_model_predict), '\n')
print(classification_report(y_test, XGB_model_predict))
print(confusion_matrix(y_test, XGB_model_predict), '\n')

### 3. CatBoost

Since XGBoost is performing well, we tried to run another gradient boosting method, called CatBoost, whose performance is often compared with Light GBM and XGBoost. It can in some cases, performs just as well as XGBoost, at a much faster rate (as expected from the cat family). To XGBoost's credit, it is older as was developed in 2014 compared to CatBoost in 2017.

To implement CatBoost, we will tune its *iterations* parameter, which is the maximum number of trees. We will also specify the class_weights or ratio between the 2 classes. 

In [ ]:
X = test.drop("Buy", axis=1)
y = test["Buy"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
params = {'iterations': range(50, 500, 50)}

model = CatBoostClassifier(eval_metric='AUC', learning_rate = 0.2, class_weights=[1, 18])
ctb = GridSearchCV(model, param_grid=params, cv=5, n_jobs=-1, scoring ='roc_auc', refit =True)
ctb.fit(X_train, y_train)
y_pred = ctb.predict(X_test)

In [ ]:
print(ctb.best_estimator_.get_all_params())
print(classification_report(y_test, y_pred), '\n')

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
(tn, fp, fn, tp)

### 4. Light GBM

We apply the Light GBM method to compare its performance to XGBoost and CatBoost. It differs from XGBoost in which it construct trees using Gradient-Based One-Sided Sampling, which is a novel approach to sampling method and as it brings down the complexity in training the decision trees, it subsequently makes the algorithm much faster to run. 

To implement LightGBM, we will tune 2 of its parameters:
1. *n_estimators*: number of boosted trees to fit
2. *boosting_type*: boosting types
    * 'gbdt': gradient boosting decision tree   
    * 'dart': dropouts meet multiple additive regression trees    
    * 'goss': gradient-based one-side sampling  
    * 'rf': random forest

In [ ]:
X = test.drop("Buy", axis=1)
y = test["Buy"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
params = {'n_estimators': range(50, 500, 50),
         'boosting_type': ['gbdt', 'dart', 'goss', 'rf']}

model = LGBMClassifier(objective='binary', learning_rate = 0.2, scale_pos_weight=18)
lgbm = GridSearchCV(model, param_grid=params, cv=5, n_jobs=-1, scoring ='roc_auc', refit =True)
lgbm.fit(X_train, y_train)
y_pred = lgbm.predict(X_test)

In [ ]:
print(lgbm.best_estimator_)
print(classification_report(y_test, y_pred), '\n')

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
(tn, fp, fn, tp)

### 5. Neural Networks

Here, we tried Neural Networks method, which is superior at capturing complex relationships between varibles by searching through different models in its black box to find patterns. In its black box, there can be one or more linear or non-linear hidden layers. 

We will apply SMOTE to the data as there is no parameter to adjust for the imbalanced in data in *MLPClassifier*. Additionally, we need to normalise the data before running neural networks as it is sensitive to feature scaling. We also specify that it will train using 2 hidden layers as 1 or 2 should give a decent performance for most problems, and 14 as number of neurons as we have 14 features in our dataset. 

In [ ]:
X = test.drop("Buy", axis=1)
y = test["Buy"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
os = SMOTE(random_state = 42, n_jobs=-1)
columns = X_train.columns

os_data_X, os_data_y = os.fit_resample(X_train, y_train)
os_data_X = pd.DataFrame(data = os_data_X,columns = columns)
os_data_y = pd.DataFrame(data = os_data_y,columns = ['Buy'])
scaler = StandardScaler()
NN_Xtrain = scaler.fit_transform(os_data_X)
NN_Xtest = scaler.transform(X_test)

mlp = MLPClassifier(hidden_layer_sizes = (14, 14), max_iter = 100, solver='adam', alpha=0.1, activation='logistic') 
#2 hidden layers and 'adam' optimizer works well for large dataset
mlp.fit(NN_Xtrain, os_data_y)

In [ ]:
y_preds = mlp.predict(NN_Xtest)
print(classification_report(y_test, y_preds))
tn, fp, fn, tp = confusion_matrix(y_test, y_preds).ravel()
print((tn, fp, fn, tp), '\n')

### 6. Support Vector Machine (SVM)

Here, we tried a non-probabilistic classifier called Support Vector Machine (SVM). This method is effective in high dimensional space and can control overfitting using its regularisation term. Additionally, it can use different kernel tricks, like polynomial or radial basis function, to transform non-linear problems to linearly separable cases. 

*SVC()* from sklearn.svm package only works well for dataset with tens of thousands observations. Since our sample has over 100,000 observations, we will need to use *SGDClassifier()* or *LinearSVC()* instead to implement SVM. Here, we will use *SGDClassifier()* as it is more flexible and faster in some cases than *LinearSVC()*. 

We will explore different kernel approximations: linear and polynomial as they seem to work well from the results below, thus it's highly unlikely that radial basis function will be a good fit. Since it will take a longer time to tune the parameters using *GridSearchCV*, we will specify the parameters directly as below:
1. *max_iter*: the default maximum number of passes over training data is 1000. However, in the interest of time and computer limitations, we will use 100 
2. *loss*: we used hinge as it is a loss function typically used for training classifiers
3. *penalty*: we used l1 as our regularisation term as it might bring sparsity in our model by shrinking some feature coefficients to 0, which helps reducing its complexity while still reducing the model's variance
4. *class_weight*: even though our actual ratio is 1:18, after testing with a few weights, we found that 1:22 gives us better performance results, which is slightly higher than the original ratio
5. *alpha*: given that the default is 0.0001 and the higher the value, the stronger the regularisation, we chose alpha value at 0.01 so that it will penalises more when there is a misclassification

In [ ]:
X = test.drop("Buy", axis=1)
y = test["Buy"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
scaler = StandardScaler()
SVM_Xtrain = scaler.fit_transform(X_train)
SVM_Xtest = scaler.transform(X_test)

svm = SGDClassifier(max_iter=100, loss='hinge', n_jobs=-1, penalty='l1', alpha = 0.01, class_weight={0: 1, 1: 22})
#adjusting class weight to get better balance between recall rates for class 0 and class 1 
svm.fit(SVM_Xtrain, y_train)
y_preds = svm.predict(SVM_Xtest)
print(classification_report(y_test, y_preds))
tn, fp, fn, tp = confusion_matrix(y_test, y_preds).ravel()
print((tn, fp, fn, tp), '\n')

Here, we will transform the data to 2nd degree polynomial kernel before running *SGDClassifier* as 3rd degree polynomial did not fit well with the data.

In [ ]:
poly_feature = PolynomialCountSketch(degree = 2, n_components = 140) #optimal n_components = 10*n_features
SVM_Xtrain = poly_feature.fit_transform(X_train)
SVM_Xtest = poly_feature.transform(X_test)

In [ ]:
svm = SGDClassifier(max_iter=100, loss='hinge', n_jobs=-1, penalty='l1', alpha = 0.01, class_weight={0: 1, 1: 20})
#adjust alpha to vary recall rates 
svm.fit(SVM_Xtrain, y_train)
y_preds = svm.predict(SVM_Xtest)
print(classification_report(y_test, y_preds))
tn, fp, fn, tp = confusion_matrix(y_test, y_preds).ravel()
print((tn, fp, fn, tp), '\n')

### 7. Stacking Classifier

Stacking is an ensemble method that combines multiple base classification models predictions, then it uses a blending model to combine the prediction of the base models. Stacking can harness the capabilities of different models and improve performance than any single model in the ensemble. The base models can be complex and diverse, while the blending model is often simple as it only needs to provide a smooth interpretation of the predictions output by the base models. 

Here, we tried a combinations of models to see if we can improve their performances. SVM, CatBoost and Logistic Regression have highest performance from the results earlier. CatBoost was chosen over Light GBM and XGBoost even though they all have the same performance because CatBoost takes the shortest time to train. 

In [ ]:
X = test.drop("Buy", axis=1)
y = test["Buy"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
estimators = [('svm', make_pipeline(StandardScaler(), 
                                    SGDClassifier(max_iter=100, loss='hinge', n_jobs=-1, 
                                                  penalty='l1', alpha = 0.01, class_weight={0: 1, 1: 22}))),
              ('ctb', CatBoostClassifier(eval_metric='AUC', learning_rate = 0.2, class_weights=[1, 18], iterations=150)),
             ('lg', make_pipeline(StandardScaler(), LogisticRegression(C=1000, solver='saga', class_weight={0: 1, 1:22})))]
clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(), cv=5, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

### Summary

As observed above, Logistic Regression, XGBoost, CatBoost, Light GBM, and SVM did well in our sample dataset. Even though Logistic Regression did not do as well as the others mentioned, we will still implement it on our full dataset as it is easy to implement while giving considerably good results. 